In [1]:
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
!df -h /content | tail -1
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/wnn_connectome_starterpack/datasets/train.parquet'):
    !cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null
!ls -lh /content/wnn_connectome_starterpack/datasets
!df -h /content | tail -1

GPU: Tesla T4
overlay         113G   43G   71G  38% /
total 33G
-rw-r--r-- 1 501 staff  28G Dec 31  2019 train.parquet
-rw-r--r-- 1 501 staff  88M Dec 31  2019 valid_mask.parquet
-rw-r--r-- 1 501 staff 5.3G Dec 31  2019 valid.parquet
overlay         113G   76G   38G  67% /


Запуск первых 4 ячеек достаточно, чтобы работали любые остальные

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch
os.makedirs('/content/drive/MyDrive/wnn', exist_ok=True); os.makedirs('/content/data', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU: Runtime -> Change runtime type -> T4')

Mounted at /content/drive
GPU: Tesla T4


In [3]:
%%writefile /content/drive/MyDrive/wnn/lib.py
import os, sys, time, subprocess
import numpy as np, torch, torch.nn as nn, pyarrow.parquet as pq

dev = 'cuda'
L, NF, WARM, SAT, U3 = 20000, 112, 99, 99, 0.6569
ROOT = '/content/wnn_connectome_starterpack/datasets'
CKPT_DIR = '/content/drive/MyDrive/wnn' if os.path.isdir('/content/drive/MyDrive/wnn') else '/content'
UN = np.array([0.1254, 0.2413, 0.4685, 0.6569])
VALMIX = np.array([0.2125, 0.1853, 0.2301, 0.3721])
post = np.zeros(L, bool); post[WARM:] = True

# ---------- модели ----------
class Net(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.gru = nn.GRU(NF, h, num_layers=2, batch_first=True)
        self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h)
        return self.head(y), h

class Net2(nn.Module):                      # то же, но с произвольным числом входов
    def __init__(self, nin, h=128):
        super().__init__()
        self.gru = nn.GRU(nin, h, num_layers=2, batch_first=True)
        self.head = nn.Linear(h, 2)
    def forward(self, x, h=None):
        y, h = self.gru(x, h)
        return self.head(y), h

class Step(nn.Module):                      # обёртка для экспорта в ONNX (интерфейс как у baseline)
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, features, hidden_0, hidden_1):
        o, h = self.net(features, torch.cat([hidden_0, hidden_1], 0))
        return o, h[0:1], h[1:2]
StepH = Step

# ---------- данные ----------
def build(src, dst, n, seed, with_mask):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet')
    meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
    feat = [c for c in pf.schema_arrow.names if c not in meta]
    assert len(feat) == NF
    idx = np.sort(np.random.default_rng(seed).choice(pf.num_row_groups, n, replace=False))
    X = np.lib.format.open_memmap(f'/content/data/{dst}_X.npy', 'w+', np.float32, (n, L, NF))
    Y = np.zeros((n, L, 2), np.float32); M = np.zeros((n, L), bool) if with_mask else None
    t = time.time()
    for k, i in enumerate(idx):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'] + (['is_scored'] if with_mask else []))
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat])
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if with_mask: M[k] = tb['is_scored'].to_numpy()
        if k % 500 == 0: print(dst, k, f'{time.time() - t:.0f} с', flush=True)
    X.flush(); np.save(f'/content/data/{dst}_Y.npy', Y)
    if with_mask: np.save(f'/content/data/{dst}_M.npy', M)

def fam_of(Y, verbose=True):                # семейство = ближайшая единица к наименьшему ненулевому |t0|
    a = np.abs(np.clip(Y[..., 0], -2, 2)).astype(np.float64)
    mn = np.where(a > 0, a, 9.0).min(1)
    f = np.abs(mn[:, None] - UN[None]).argmin(1)
    if verbose: print('макс. отклонение от единицы:', np.abs(mn - UN[f]).max().round(4), '| классы:', np.bincount(f, minlength=4))
    return f

# ---------- обучение и метрика ----------
def loss_fn(p, y, mask):
    y = y.clamp(-2, 2)
    return (y.abs() * (p - y) ** 2)[mask].mean()

@torch.no_grad()
def evaluate(model, X, Y, M, bs=50, ch=2000):
    model.eval()
    S = {k: torch.zeros(2, 6, dtype=torch.float64, device=dev) for k in ('все строки', 'is_scored')}
    pm = torch.from_numpy(post).to(dev)
    for s in range(0, len(X), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)
        y = torch.from_numpy(np.ascontiguousarray(Y[s:s + bs])).to(dev).clamp(-2, 2)
        m_sc = torch.from_numpy(np.ascontiguousarray(M[s:s + bs])).to(dev)
        h, outs = None, []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); outs.append(o)
        p = torch.cat(outs, 1).clamp(-2, 2)
        for name, mask in (('все строки', pm[None].expand_as(m_sc)), ('is_scored', m_sc & pm[None])):
            for j in (0, 1):
                yy, pp = y[..., j][mask].double(), p[..., j][mask].double(); a = yy.abs()
                S[name][j] += torch.stack([a.sum(), (a*yy).sum(), (a*pp).sum(), (a*yy*yy).sum(), (a*pp*pp).sum(), (a*yy*pp).sum()])
    out = {}
    for k, T in S.items():
        sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
        cov = T[:, 5] / sw - my * mp
        den = (T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2)
        c = torch.where(den > 1e-16, cov / den.clamp_min(1e-16).sqrt(), torch.zeros_like(cov)).clamp(-1, 1)
        out[k] = (c.mean().item(), c.cpu().numpy().round(3))
    model.train(); return out

def train(model, epochs, bs=32, ch=1000, lr=2e-3, tag='gru'):     # использует глобальные Xtr, Ytr, Xva, Yva, Mva
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    steps_total = epochs * (len(Xtr) // bs) * (L // ch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps_total, pct_start=0.05)
    rng = np.random.default_rng(0); best = -1
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(len(Xtr)); run = []
        for g in range(len(Xtr) // bs):
            idx = np.sort(order[g * bs:(g + 1) * bs])
            x = torch.from_numpy(np.ascontiguousarray(Xtr[idx])).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev)
            h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach()
                mask = (torch.arange(t, t + ch, device=dev) >= WARM)[None].expand(len(idx), -1)
                loss = loss_fn(o, y[:, t:t + ch], mask)
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
                run.append(loss.item())
        r = evaluate(model, Xva, Yva, Mva)
        print(f'эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | '
              f'valid все строки {r["все строки"][0]:.3f} {r["все строки"][1]} | is_scored {r["is_scored"][0]:.3f} {r["is_scored"][1]}', flush=True)
        if r['is_scored'][0] > best:
            best = r['is_scored'][0]; torch.save(model.state_dict(), f'{CKPT_DIR}/{tag}_best.pt')
    return model

# ---------- анализ по семействам и валидационной смеси ----------
@torch.no_grad()
def predict(model, X, idx, ch=2000, bs=49):
    model.eval(); out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.ascontiguousarray(X[idx[s:s + bs]])).to(dev); h = None; o_ = []
        for t in range(0, L, ch):
            o, h = model(x[:, t:t + ch], h); o_.append(o)
        out.append(torch.cat(o_, 1).clamp(-2, 2).cpu().numpy())
    return np.concatenate(out)

def sums_np(P, Y, M):
    post_m = np.broadcast_to(post, M.shape); out = {}
    for name, m in (('все строки', post_m), ('is_scored', M & post_m)):
        T = np.zeros((2, 6))
        for j in (0, 1):
            y = np.clip(Y[..., j][m], -2, 2).astype(np.float64); p = np.clip(P[..., j][m], -2, 2).astype(np.float64); a = np.abs(y)
            T[j] = [a.sum(), (a*y).sum(), (a*p).sum(), (a*y*y).sum(), (a*p*p).sum(), (a*y*p).sum()]
        out[name] = T
    return out

def corr(T):
    sw = T[:, 0]; my, mp = T[:, 1] / sw, T[:, 2] / sw
    cov = T[:, 5] / sw - my * mp
    den = (T[:, 3] / sw - my ** 2) * (T[:, 4] / sw - mp ** 2)
    return float(np.mean(np.where(den > 1e-16, cov / np.sqrt(np.maximum(den, 1e-16)), 0.0)))

def ens_mix(models):                        # скор по валидационной смеси семейств; нужны Xva, Yva, Mva, fam_v, have
    m = VALMIX / have; Sm = {}
    for f in range(4):
        ix = np.where(fam_v == f)[0]
        P = np.mean([predict(mod, Xva, ix) for mod in models], axis=0)
        Sm[f] = sums_np(P, Yva[ix], Mva[ix])
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 3) for k in ('все строки', 'is_scored')}
mix_score = lambda model: ens_mix([model])

def code3(y):                               # код k в единицах семейства 4; насыщение = ±99
    k = np.round(y / U3).astype(np.int32); k[y >= 2 - 1e-6] = SAT; k[y <= -2 + 1e-6] = -SAT; return k

def wc(y, p, m):                            # взвешенная корреляция по маске m
    y, p = y[m].astype(np.float64), p[m].astype(np.float64); w = np.abs(y); sw = w.sum()
    my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    cov, vy, vp = (w * (y - my) * (p - mp)).sum(), (w * (y - my) ** 2).sum(), (w * (p - mp) ** 2).sum()
    return cov / np.sqrt(vy * vp) if vp > 1e-12 else 0.0

Writing /content/drive/MyDrive/wnn/lib.py


In [4]:
%run -i /content/drive/MyDrive/wnn/lib.py

need = {'train': (600, 0, False), 'train3k': (3000, 0, False), 'valid300': (300, 1, True)}
def done(n): return all(os.path.exists(f'/content/data/{n}_{s}.npy') for s in (['X', 'Y'] + (['M'] if n.startswith('valid') else [])))
missing = [n for n in need if not done(n)]
print('нет файлов для:', missing or 'всё на месте')
if missing:
    if not os.path.exists(f'{ROOT}/train.parquet'):
        print('качаю данные (около 11 минут)...')
        subprocess.run("curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | "
                       "tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null",
                       shell=True, cwd='/content')
    for n in missing:
        src = 'valid' if n.startswith('valid') else 'train'
        build(src, n, *need[n])
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)

нет файлов для: ['train', 'train3k', 'valid300']
train 0 1 с
train 500 42 с
train3k 0 0 с
train3k 500 39 с
train3k 1000 76 с
train3k 1500 110 с
train3k 2000 150 с
train3k 2500 200 с
valid300 0 0 с
overlay         113G  109G  4.4G  97% /



In [5]:
ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')
Xtr, Ytr = ld('train_X'), np.load('/content/data/train_Y.npy'); Y6 = Ytr          # 600 последовательностей
Y3k = np.load('/content/data/train3k_Y.npy')                                       # 3000 (X: ld('train3k_X'))
Xva = ld('valid300_X'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy'); Y3 = Yva
fam_v = fam_of(Yva); have = np.bincount(fam_v, minlength=4) / len(fam_v)
f3k = fam_of(Y3k)

paths = [f'{CKPT_DIR}/gru_w3k_best.pt', '/content/gru_w3k_best.pt', '/content/drive/MyDrive/wnn/gru_w3k_best.pt']
CKPT = next((p for p in paths if os.path.exists(p)), None)
if CKPT:
    net128 = Net(128).to(dev); net128.load_state_dict(torch.load(CKPT, map_location=dev)); net128.eval()
    print('h=128 с', CKPT, '->', mix_score(net128), '(раньше было 0.487 / 0.653)')
else:
    print('чекпойнта gru_w3k_best.pt нет. Переобучить (около 19 минут):\n'
          "  Xtr, Ytr = ld('train3k_X'), Y3k; torch.manual_seed(0); net128 = train(Net().to(dev), epochs=8, tag='gru_w3k')")

макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]
макс. отклонение от единицы: 0.0 | классы: [ 387  375  955 1283]
чекпойнта gru_w3k_best.pt нет. Переобучить (около 19 минут):
  Xtr, Ytr = ld('train3k_X'), Y3k; torch.manual_seed(0); net128 = train(Net().to(dev), epochs=8, tag='gru_w3k')


Запускать, если чекпойнта gru_w3k_best.pt нет. Переобучить (около 19 минут)

In [7]:
print('чекпойнты сохраняются в:', CKPT_DIR)                       # должно быть /content/drive/MyDrive/wnn
Xtr, Ytr = ld('train3k_X'), Y3k
Xva = ld('valid300_X'); Yva, Mva = Y3, np.load('/content/data/valid300_M.npy')
torch.manual_seed(0)
net_last = train(Net().to(dev), epochs=8, tag='gru_w3k')         # лучшая эпоха по is_scored на valid300 пишется в gru_w3k_best.pt

net128 = Net(128).to(dev); net128.load_state_dict(torch.load(f'{CKPT_DIR}/gru_w3k_best.pt', map_location=dev)); net128.eval()
print('лучшая эпоха, валидационная смесь:', mix_score(net128), '| последняя эпоха:', mix_score(net_last), '| раньше было 0.487 / 0.653')

чекпойнты сохраняются в: /content/drive/MyDrive/wnn


/content/drive/MyDrive/wnn/lib.py:75: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)


эпоха 1/8 | 153 с | loss 1.0530 | valid все строки 0.447 [0.444 0.45 ] | is_scored 0.647 [0.643 0.652]
эпоха 2/8 | 155 с | loss 1.0012 | valid все строки 0.471 [0.47  0.473] | is_scored 0.653 [0.65  0.656]
эпоха 3/8 | 155 с | loss 0.9895 | valid все строки 0.476 [0.48  0.472] | is_scored 0.652 [0.65  0.654]
эпоха 4/8 | 154 с | loss 0.9794 | valid все строки 0.477 [0.472 0.482] | is_scored 0.658 [0.654 0.663]
эпоха 5/8 | 153 с | loss 0.9695 | valid все строки 0.486 [0.485 0.487] | is_scored 0.662 [0.657 0.666]
эпоха 6/8 | 153 с | loss 0.9497 | valid все строки 0.489 [0.491 0.486] | is_scored 0.666 [0.661 0.671]
эпоха 7/8 | 153 с | loss 0.9308 | valid все строки 0.489 [0.49  0.488] | is_scored 0.666 [0.662 0.67 ]
эпоха 8/8 | 152 с | loss 0.9185 | valid все строки 0.488 [0.489 0.487] | is_scored 0.665 [0.661 0.67 ]
лучшая эпоха, валидационная смесь: {'все строки': 0.488, 'is_scored': 0.654} | последняя эпоха: {'все строки': 0.487, 'is_scored': 0.653} | раньше было 0.487 / 0.653


Проверка можно ли ужать данные

In [8]:
class AsF16:
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i]).astype(np.float16).astype(np.float32)
Xva_orig = Xva
print('float32-вход:', mix_score(net128))
Xva = AsF16(Xva_orig); print('float16-вход:', mix_score(net128)); Xva = Xva_orig

float32-вход: {'все строки': 0.488, 'is_scored': 0.654}
float16-вход: {'все строки': 0.488, 'is_scored': 0.654}


Пересобираем посл-ти на float16

In [10]:
!du -h --max-depth=2 /content 2>/dev/null | sort -h | tail -12
!ls -lh /content/data
!df -h /content | tail -1

12K	/content/drive/.Trash-0
80K	/content/drive/.Encrypted
92K	/content/.config/logs
148K	/content/.config
1.5M	/content/wnn_connectome_starterpack/baseline
55M	/content/sample_data
96M	/content/drive
96M	/content/drive/MyDrive
6.0G	/content/data
33G	/content/wnn_connectome_starterpack
33G	/content/wnn_connectome_starterpack/datasets
39G	/content
total 6.0G
-rw-r--r-- 1 root root 458M Nov  1 18:38 train3k_Y.npy
-rw-r--r-- 1 root root 3.8G Nov  1 19:11 trainfull_X16.npy
-rw-r--r-- 1 root root  92M Nov  1 18:34 train_Y.npy
-rw-r--r-- 1 root root 5.8M Nov  1 18:39 valid300_M.npy
-rw-r--r-- 1 root root 2.6G Nov  1 18:39 valid300_X.npy
-rw-r--r-- 1 root root  46M Nov  1 18:39 valid300_Y.npy
overlay         113G  107G  6.5G  95% /


In [11]:
print([l.split()[-2:] for l in open('/proc/self/maps') if 'deleted' in l][:3])

[['/dev/zero', '(deleted)'], ['/content/data/train3k_X.npy', '(deleted)'], ['/dev/zero', '(deleted)']]


In [13]:
print([l.split()[-2:] for l in open('/proc/self/maps') if 'deleted' in l][:3])
import gc; Xtr = None; Xva_orig = None; gc.collect()
!df -h /content | tail -1

[['/dev/zero', '(deleted)'], ['/dev/zero', '(deleted)'], ['/dev/zero', '(deleted)']]
overlay         113G   82G   32G  73% /


In [14]:
!rm -f /content/data/trainfull_X16.npy /content/data/trainfull_Y.npy
!df -h /content | tail -1

def build16(src, dst, n, seed):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet')
    meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
    feat = [c for c in pf.schema_arrow.names if c not in meta]; assert len(feat) == NF
    idx = np.sort(np.random.default_rng(seed).choice(pf.num_row_groups, n, replace=False))
    X = np.lib.format.open_memmap(f'/content/data/{dst}_X16.npy', 'w+', np.float16, (n, L, NF))
    Y = np.zeros((n, L, 2), np.float32); t = time.time()
    for k, i in enumerate(idx):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'])
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat]).astype(np.float16)
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if k % 1000 == 0: print(dst, k, f'{time.time() - t:.0f} с', flush=True)
    X.flush(); np.save(f'/content/data/{dst}_Y.npy', Y)

build16('train', 'trainfull', 5000, 0)
!df -h /content | tail -1

overlay         113G   82G   32G  73% /
trainfull 0 0 с
trainfull 1000 77 с
trainfull 2000 157 с
trainfull 3000 238 с
trainfull 4000 326 с
overlay         113G  103G  9.9G  92% /


In [15]:
class V16:                                   # читает float16 с диска, отдаёт float32
    def __init__(s, X): s.X = X
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return np.asarray(s.X[i]).astype(np.float32)

Xtr = V16(np.load('/content/data/trainfull_X16.npy', mmap_mode='r'))
Ytr = np.load('/content/data/trainfull_Y.npy')
fam_tr = fam_of(Ytr)
print('доли семейств train:', np.round(np.bincount(fam_tr, minlength=4) / len(fam_tr), 3), '| valid:', np.round(VALMIX, 3))

Xva = ld('valid300_X'); Yva = Y3; Mva = np.load('/content/data/valid300_M.npy')
torch.manual_seed(0)
net_full = train(Net().to(dev), epochs=5, tag='gru_full')
net_best = Net(128).to(dev); net_best.load_state_dict(torch.load(f'{CKPT_DIR}/gru_full_best.pt', map_location=dev)); net_best.eval()
print('лучшая эпоха:', mix_score(net_best), '| последняя:', mix_score(net_full), '| на 3000 было 0.488 / 0.654 (лучшая эпоха)')

макс. отклонение от единицы: 0.0 | классы: [ 612  593 1596 2199]
доли семейств train: [0.122 0.119 0.319 0.44 ] | valid: [0.212 0.185 0.23  0.372]
эпоха 1/5 | 224 с | loss 1.0500 | valid все строки 0.457 [0.454 0.461] | is_scored 0.650 [0.648 0.652]
эпоха 2/5 | 223 с | loss 1.0090 | valid все строки 0.482 [0.485 0.478] | is_scored 0.661 [0.659 0.662]
эпоха 3/5 | 218 с | loss 0.9943 | valid все строки 0.489 [0.491 0.486] | is_scored 0.665 [0.661 0.67 ]
эпоха 4/5 | 215 с | loss 0.9771 | valid все строки 0.491 [0.491 0.49 ] | is_scored 0.667 [0.662 0.673]
эпоха 5/5 | 213 с | loss 0.9619 | valid все строки 0.491 [0.492 0.49 ] | is_scored 0.668 [0.665 0.671]
лучшая эпоха: {'все строки': 0.491, 'is_scored': 0.656} | последняя: {'все строки': 0.491, 'is_scored': 0.656} | на 3000 было 0.488 / 0.654 (лучшая эпоха)


In [16]:
!rm -f /content/wnn_connectome_starterpack/datasets/train.parquet
!df -h /content | tail -1

pfv = pq.ParquetFile(f'{ROOT}/valid.parquet'); assert pfv.num_row_groups == 1873
held = set(np.random.default_rng(1).choice(1873, 300, replace=False).tolist())    # те же 300, что в valid300
rest = np.array([i for i in range(1873) if i not in held]); print('валидационных в обучение:', len(rest))

meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
featv = [c for c in pfv.schema_arrow.names if c not in meta]; assert len(featv) == NF
Xr = np.lib.format.open_memmap('/content/data/validrest_X16.npy', 'w+', np.float16, (len(rest), L, NF))
Yr = np.zeros((len(rest), L, 2), np.float32)
for k, i in enumerate(rest):
    tb = pfv.read_row_group(int(i), columns=featv + ['t0', 't1'])
    Xr[k] = np.column_stack([tb[c].to_numpy() for c in featv]).astype(np.float16)
    Yr[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
Xr.flush()

class Cat:                                    # склеивает два набора; индексы приходят отсортированными
    def __init__(s, A, B): s.A, s.B, s.n = A, B, len(A)
    def __len__(s): return s.n + len(s.B)
    def __getitem__(s, idx):
        idx = np.asarray(idx); a, b = idx[idx < s.n], idx[idx >= s.n] - s.n
        return np.concatenate([p for p in ((s.A[a] if len(a) else None), (s.B[b] if len(b) else None)) if p is not None])

Xtr = Cat(V16(np.load('/content/data/trainfull_X16.npy', mmap_mode='r')), V16(np.load('/content/data/validrest_X16.npy', mmap_mode='r')))
Ytr = np.concatenate([np.load('/content/data/trainfull_Y.npy'), Yr])
Xva = ld('valid300_X'); Yva = Y3; Mva = np.load('/content/data/valid300_M.npy')
print('обучающих последовательностей:', len(Xtr))

torch.manual_seed(0)
net_v = train(Net().to(dev), epochs=5, tag='gru_full_val')
print('последняя эпоха:', mix_score(net_v), '| только train (5000):', '0.491 / 0.656')

overlay         113G  103G  9.9G  92% /
валидационных в обучение: 1573
обучающих последовательностей: 6573
эпоха 1/5 | 224 с | loss 0.9634 | valid все строки 0.468 [0.47  0.465] | is_scored 0.650 [0.646 0.653]
эпоха 2/5 | 226 с | loss 0.9268 | valid все строки 0.484 [0.483 0.485] | is_scored 0.662 [0.657 0.668]
эпоха 3/5 | 225 с | loss 0.9155 | valid все строки 0.486 [0.49  0.482] | is_scored 0.658 [0.658 0.658]
эпоха 4/5 | 225 с | loss 0.9019 | valid все строки 0.499 [0.5   0.499] | is_scored 0.669 [0.665 0.673]
эпоха 5/5 | 223 с | loss 0.8868 | valid все строки 0.499 [0.5   0.498] | is_scored 0.670 [0.666 0.673]
последняя эпоха: {'все строки': 0.498, 'is_scored': 0.657} | только train (5000): 0.491 / 0.656


In [17]:
class Perm:
    def __init__(s, X, cols, perm): s.X, s.cols, s.perm = X, cols, perm
    def __len__(s): return len(s.X)
    def __getitem__(s, idx):
        idx = np.asarray(idx); b = np.array(s.X[idx])
        b[:, :, s.cols] = np.asarray(s.X[s.perm[idx]])[:, :, s.cols]
        return b

Xva0 = ld('valid300_X'); Yva, Mva = Y3, np.load('/content/data/valid300_M.npy')
perm = np.arange(len(Xva0))
for f in range(4):
    ix = np.where(fam_v == f)[0]; perm[ix] = np.random.default_rng(f).permutation(ix)
groups = {'i0 цены (0-21)': range(0, 22), 'i0 объёмы (22-43)': range(22, 44), 'i0 сделки (44-51)': range(44, 52),
          'i1 цены (52-73)': range(52, 74), 'i1 объёмы (74-95)': range(74, 96), 'i1 сделки (96-103)': range(96, 104)}
groups.update({f'a{k}': [104 + k] for k in range(8)})
Xva = Xva0; print('без перестановки:', mix_score(net_v))
for name, cols in groups.items():
    Xva = Perm(Xva0, list(cols), perm); r = mix_score(net_v)
    print(f'{name:22s} {r["все строки"]:.3f} / {r["is_scored"]:.3f}')
Xva = Xva0

без перестановки: {'все строки': 0.498, 'is_scored': 0.657}
i0 цены (0-21)         0.455 / 0.619
i0 объёмы (22-43)      0.401 / 0.559
i0 сделки (44-51)      0.470 / 0.637
i1 цены (52-73)        0.498 / 0.655
i1 объёмы (74-95)      0.496 / 0.655
i1 сделки (96-103)     0.476 / 0.631
a0                     0.497 / 0.655
a1                     0.498 / 0.657
a2                     0.499 / 0.655
a3                     0.499 / 0.656
a4                     0.498 / 0.657
a5                     0.498 / 0.656
a6                     0.483 / 0.649
a7                     0.495 / 0.654


In [18]:
class NetX(nn.Module):
    def __init__(s, mode, h=128):
        super().__init__(); s.mode = mode; nin = NF
        if mode == 'flags': nin += 3
        if mode == 'mlp':
            s.mlp = nn.Sequential(nn.Linear(NF, 128), nn.GELU(), nn.Linear(128, 128)); nin += 128
        s.gru = nn.GRU(nin, h, num_layers=2, batch_first=True); s.head = nn.Linear(h, 2)
    def forward(s, x, h=None):
        z = [x]
        if s.mode == 'flags': z.append((x[..., 109:112] < -5.19).float())     # a5, a6, a7 на заглушке
        if s.mode == 'mlp': z.append(s.mlp(x))
        y, h = s.gru(torch.cat(z, -1), h); return s.head(y), h

Xtr = V16(np.load('/content/data/trainfull_X16.npy', mmap_mode='r')[:600]); Ytr = np.load('/content/data/trainfull_Y.npy')[:600]
Xva = ld('valid300_X'); res = {}
for mode in ('none', 'flags', 'mlp'):
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(NetX(mode).to(dev), epochs=8, tag=f'x_{mode}_{seed}')
        res[(mode, seed)] = mix_score(m); print('>>>', mode, seed, res[(mode, seed)], flush=True)

эпоха 1/8 | 28 с | loss 1.1261 | valid все строки 0.428 [0.432 0.424] | is_scored 0.600 [0.61  0.589]
эпоха 2/8 | 31 с | loss 1.0577 | valid все строки 0.436 [0.418 0.455] | is_scored 0.632 [0.62  0.643]
эпоха 3/8 | 26 с | loss 1.0244 | valid все строки 0.449 [0.446 0.452] | is_scored 0.641 [0.637 0.644]
эпоха 4/8 | 30 с | loss 1.0098 | valid все строки 0.461 [0.459 0.463] | is_scored 0.648 [0.646 0.651]
эпоха 5/8 | 30 с | loss 0.9865 | valid все строки 0.469 [0.469 0.468] | is_scored 0.653 [0.647 0.659]
эпоха 6/8 | 31 с | loss 0.9702 | valid все строки 0.469 [0.47  0.468] | is_scored 0.653 [0.649 0.656]
эпоха 7/8 | 31 с | loss 0.9440 | valid все строки 0.468 [0.469 0.468] | is_scored 0.654 [0.651 0.658]
эпоха 8/8 | 31 с | loss 0.9350 | valid все строки 0.468 [0.47  0.465] | is_scored 0.654 [0.651 0.658]
>>> none 0 {'все строки': 0.468, 'is_scored': 0.642}
эпоха 1/8 | 31 с | loss 1.1204 | valid все строки 0.428 [0.437 0.419] | is_scored 0.600 [0.612 0.587]
эпоха 2/8 | 33 с | loss 1.059

In [19]:
if 'loss_orig' not in globals(): loss_orig = loss_fn
AUX, LAM = False, 0.5

def loss_fn(p, y, mask):
    y = y.clamp(-2, 2)
    main = (y.abs() * (p[..., :2] - y) ** 2)[mask].mean()
    if not AUX: return main
    side = (y[..., 0] < y[..., 1]).float()                  # 1: нижний t0; при y0 == y1 вес нулевой
    w = (y[..., 0] - y[..., 1]).abs()
    bce = nn.functional.binary_cross_entropy_with_logits(p[..., 2], side, reduction='none')
    return main + LAM * (w * bce)[mask].mean()

class NetA(nn.Module):
    def __init__(s, h=128, drop=False):
        super().__init__(); s.drop = drop
        s.gru = nn.GRU(NF, h, num_layers=2, batch_first=True); s.head = nn.Linear(h, 3)
    def forward(s, x, h=None):
        if s.drop: x = x.clone(); x[..., 52:96] = 0
        y, h = s.gru(x, h); return s.head(y), h

Xtr = V16(np.load('/content/data/trainfull_X16.npy', mmap_mode='r')[:600]); Ytr = np.load('/content/data/trainfull_Y.npy')[:600]
Xva = ld('valid300_X'); Yva = Y3; Mva = np.load('/content/data/valid300_M.npy')
for name, drop, aux in (('вспомогательная голова', False, True), ('без стакана i1', True, False)):
    AUX = aux
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(NetA(drop=drop).to(dev), epochs=8, tag=f'a_{seed}')
        print('>>>', name, seed, mix_score(m), flush=True)
AUX = False; loss_fn = loss_orig

эпоха 1/8 | 32 с | loss 1.5448 | valid все строки 0.437 [0.446 0.428] | is_scored 0.613 [0.625 0.601]
эпоха 2/8 | 34 с | loss 1.4529 | valid все строки 0.439 [0.423 0.454] | is_scored 0.637 [0.628 0.647]
эпоха 3/8 | 32 с | loss 1.4121 | valid все строки 0.438 [0.435 0.44 ] | is_scored 0.628 [0.632 0.624]
эпоха 4/8 | 33 с | loss 1.3939 | valid все строки 0.463 [0.461 0.465] | is_scored 0.653 [0.65  0.656]
эпоха 5/8 | 31 с | loss 1.3632 | valid все строки 0.470 [0.477 0.464] | is_scored 0.655 [0.65 0.66]
эпоха 6/8 | 27 с | loss 1.3432 | valid все строки 0.472 [0.473 0.471] | is_scored 0.656 [0.653 0.66 ]
эпоха 7/8 | 31 с | loss 1.3085 | valid все строки 0.470 [0.472 0.468] | is_scored 0.658 [0.655 0.661]
эпоха 8/8 | 30 с | loss 1.2976 | valid все строки 0.470 [0.473 0.466] | is_scored 0.657 [0.654 0.66 ]
>>> вспомогательная голова 0 {'все строки': 0.47, 'is_scored': 0.645}
эпоха 1/8 | 33 с | loss 1.5355 | valid все строки 0.434 [0.444 0.423] | is_scored 0.607 [0.623 0.592]
эпоха 2/8 | 31